In [1]:
import subprocess
import os
import pandas as pd
import numpy as np
from core.reader import read_ansys_csv, load_experimental_data

ANSYS_EXE_PATH = r"D:\Program Files\ANSYS Inc\ANSYS Student\v252\ansys\bin\winx64\MAPDL.exe" 
WORKING_DIR = os.getcwd()

def run_ansys_simulation(params):
    with open('chab_params.txt', 'w') as f:
        for p in params:
            f.write(f"{p}\n")

    input_file = "chab.mac"
    output_file = "ansys.out"
    
    cmd = [
        ANSYS_EXE_PATH, 
        "-b",
        "-j", "opt_run",
        "-dir", WORKING_DIR, 
        "-i", input_file, 
        "-o", output_file
    ]

    try:
        subprocess.run(cmd, check=True, capture_output=True)
    except subprocess.CalledProcessError as e:
        print("Ошибка ANSYS:", e)
        return None

    try:
        df_res = read_ansys_csv("chab.csv")
        return df_res
    except Exception as e:
        print(f"Ошибка чтения CSV: {e}")
        return None

In [2]:
real_experiment_data_folder = "."
df_exp = load_experimental_data(real_experiment_data_folder)

zero_row = pd.DataFrame(0.0, columns=df_exp.columns, index=[0])
zero_row['Time'] = 1.0
df_exp = pd.concat([zero_row, df_exp]).reset_index(drop=True)

def objective_function(params):
    """
    Считает ошибку между экспериментом и моделью Шабоша.
    params: [sig_y, c1, g1, c2, g2, c3, g3]
    """
    print(f"Simulating: {params}")
    
    df_ansys = run_ansys_simulation(params)
    
    if df_ansys is None or len(df_ansys) != len(df_exp):
        return 1e9

    mse_zz = np.mean((df_ansys['S_ZZ'] - df_exp['S_ZZ'])**2)
    mse_tt = np.mean((df_ansys['S_TT'] - df_exp['S_TT'])**2)
    mse_tz = np.mean((df_ansys['S_TZ'] - df_exp['S_TZ'])**2)
    
    total_error = mse_zz + mse_tt + mse_tz
    print(f"Error: {total_error:.2f}")
    return total_error

In [3]:
import psutil

def kill_ansys_processes():
    for proc in psutil.process_iter():
        if proc.name() in ['ANSYS.exe', 'MAPDL.exe', 'ansys.exe']:
            proc.kill()

kill_ansys_processes()

In [4]:
from scipy.optimize import differential_evolution

# Границы поиска для каждого параметра
# [sig_y, c1, g1, c2, g2, c3, g3]
bounds = [
    (200, 400),      # Sig_Y
    (1e4, 5e5),      # C1 (Жесткая кинематика)
    (100, 5000),     # gamma1 (Быстрое насыщение)
    (1e3, 5e4),      # C2
    (10, 500),       # gamma2
    (100, 1e4),      # C3
    (0, 100)         # gamma3 (Линейная часть)
]

result = differential_evolution(
    objective_function, 
    bounds, 
    strategy='best1bin', 
    maxiter=15,      # Количество поколений (увеличьте до 20-50 для точности)
    popsize=10,       # Размер популяции (увеличьте до 10-15)
    disp=True,
    polish=True,
    workers=1
)

print("Оптимальные параметры найдены:")
print(result.x)

Simulating: [3.41812291e+02 3.89500632e+05 1.82968315e+03 4.91741754e+04
 3.49353826e+02 6.47668450e+03 5.65196414e+01]
Error: 74670.22
Simulating: [  239.09464799 65368.9852074   3386.2944666   1611.1127248
   419.73776175  7678.50676003    66.06147615]
Error: 38687.56
Simulating: [3.80944290e+02 4.51843694e+05 1.37301545e+03 1.07523962e+04
 8.18836256e+01 2.81716734e+03 1.52389483e+01]
Error: 132319.92
Simulating: [3.70246564e+02 9.12862798e+04 4.46880871e+03 4.00215133e+04
 1.83024353e+02 9.03867048e+02 4.85265758e+01]
Error: 9835.67
Simulating: [  334.76780553 30052.98884891   800.0156578  23825.29805378
   480.69709649  2678.23143645    38.96716955]
Error: 11380.64
Simulating: [  314.98667631 50707.5157575   1941.4192083  32678.24720921
   356.77500275  6264.72275503    86.06223191]
Error: 8152.94
Simulating: [2.68011591e+02 1.60005545e+05 3.24566556e+03 1.82314590e+04
 9.09009736e+01 1.70389963e+03 9.02597403e+00]
Error: 8926.23
Simulating: [  257.87205726 67987.45918968  1329.76